# Chapter 2 — Probability and Statistics

Companion notebook for *The Math That Powers AI* (2nd ed.), Chapter 2.

All functions come from the `mathpowersai.probability` package module — we import them rather than redefining them. Every random draw flows through a single seeded generator, `rng = np.random.default_rng(42)`, so the whole notebook is deterministic.

**Contents**

1. Bayes' theorem and the base-rate fallacy (the medical-test example)
2. Maximum likelihood estimation for a Bernoulli parameter
3. The sample mean and the two variance estimators (1/n vs 1/(n-1))
4. The variance of the sample mean shrinks as sigma^2 / n
5. A complete Naive Bayes spam classifier

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np

In [ ]:
from mathpowersai.probability import (
    NaiveBayesClassifier,
    bayes_theorem,
    bernoulli_mle,
    mle_variance,
    sample_mean,
    sample_variance,
    simulate_sample_means,
)

# One seeded generator for the whole notebook: fully deterministic.
rng = np.random.default_rng(42)

## 1. Bayes' theorem: the medical-test example

Bayes' theorem inverts a conditional probability:

$$P(A \mid B) = \frac{P(B \mid A)\,P(A)}{P(B)}, \qquad
P(B) = P(B \mid A)\,P(A) + P(B \mid A^c)\,P(A^c).$$

The chapter's running example: a disease affects **1%** of the population (the *prior*, or base rate). A test detects it with **95%** sensitivity, $P(+ \mid \text{disease}) = 0.95$, but also fires on **10%** of healthy people, $P(+ \mid \text{healthy}) = 0.10$.

Intuition says a positive test means you probably have the disease. Bayes says otherwise: the posterior is only about **0.0876** — under 9% — because the huge population of healthy people produces far more false positives than the tiny diseased population produces true positives. Ignoring the prior like this is the *base-rate fallacy*.

In [ ]:
prior_disease = 0.01     # P(disease): the base rate
sensitivity = 0.95       # P(+ | disease)
false_positive = 0.10    # P(+ | healthy)

posterior = bayes_theorem(prior_disease, sensitivity, false_positive)
print(f"P(disease | positive test) = {posterior:.4f}")   # ~0.0876

# Same machinery, spam flavour (the chapter's running example):
# 30% of email is spam; 'free' appears in 80% of spam, 10% of ham.
p_spam = bayes_theorem(prior=0.30, likelihood=0.80,
                       false_positive_rate=0.10)
print(f"P(spam | 'free')           = {p_spam:.3f}")

## 2. Maximum likelihood estimation for a Bernoulli parameter

Given $k$ successes in $n$ independent Bernoulli($p$) trials, the log-likelihood is

$$\ell(p) = k \log p + (n - k) \log(1 - p).$$

Setting $\mathrm{d}\ell/\mathrm{d}p = k/p - (n-k)/(1-p) = 0$ gives the maximum likelihood estimate

$$\hat{p}_{\text{MLE}} = \frac{k}{n}.$$

The MLE is just the empirical frequency. This is exactly how Naive Bayes (Section 5) learns its word probabilities: if 800 of 1,000 spam emails contain "free", then $\hat{p} = 0.80$. We can also sanity-check the estimator on seeded simulated coin flips.

In [ ]:
# The chapter's example: 800 of 1000 spam emails contain "free".
p_hat = bernoulli_mle(k=800, n=1000)
print(f"800 of 1000 spam emails contain 'free' -> p_hat = {p_hat:.2f}")

# Sanity check on simulated data: flip a biased coin (p = 0.3)
# and watch p_hat = k/n home in on the truth as n grows.
p_true = 0.3
for n in (10, 100, 1000, 10000):
    flips = rng.random(n) < p_true        # Bernoulli(0.3) draws
    k = int(flips.sum())
    print(f"n={n:6d}: k={k:5d}, p_hat = {bernoulli_mle(k, n):.4f} "
          f"(true p = {p_true})")

## 3. The sample mean and the two variance estimators

For samples $x_1, \dots, x_n$ from a Gaussian, the MLE of the mean is the **sample mean**

$$\hat{\mu}_{\text{MLE}} = \bar{x} = \frac{1}{n} \sum_i x_i,$$

and the MLE of the variance divides by $n$:

$$\hat{\sigma}^2_{\text{MLE}} = \frac{1}{n} \sum_i (x_i - \bar{x})^2.$$

The MLE variance is *biased*: $\mathbb{E}[\hat{\sigma}^2_{\text{MLE}}] = \frac{n-1}{n}\sigma^2$, so it underestimates $\sigma^2$. Dividing by $n - 1$ instead gives the **unbiased sample variance** $s^2$. The two differ by exactly a factor of $n/(n-1)$ — large for tiny samples, negligible for big ones. In ML we usually just use the MLE, because we care about prediction rather than exact parameter recovery.

In [ ]:
# A small sample makes the 1/n vs 1/(n-1) gap visible.
x = rng.normal(0, 1, 10)     # 10 draws from N(0, 1), true sigma^2 = 1

xbar = sample_mean(x)
v_mle = mle_variance(x)      # divides by n
v_unb = sample_variance(x)   # divides by n - 1

print(f"n = {x.size}")
print(f"sample mean        = {xbar:.4f}")
print(f"MLE variance (1/n)       = {v_mle:.4f}")
print(f"unbiased variance (1/(n-1)) = {v_unb:.4f}")
print(f"ratio unbiased/MLE = {v_unb / v_mle:.4f} "
      f"(theory n/(n-1) = {x.size / (x.size - 1):.4f})")

# With a large sample the distinction all but disappears.
x_big = rng.normal(0, 1, 100_000)
print(f"\nn = {x_big.size}: MLE = {mle_variance(x_big):.5f}, "
      f"unbiased = {sample_variance(x_big):.5f}")

## 4. The variance of the sample mean shrinks as sigma^2 / n

If $x_1, \dots, x_n \sim \mathcal{N}(\mu, \sigma^2)$ i.i.d., then the sample mean is itself a random variable with

$$\mathbb{E}[\bar{x}] = \mu, \qquad \operatorname{Var}(\bar{x}) = \frac{\sigma^2}{n}.$$

Averaging $n$ samples shrinks the noise by a factor of $n$ — this is why more data gives more reliable estimates, and (via the Central Limit Theorem) why $\bar{x}$ looks Gaussian regardless of the sample size, once $n$ is moderately large.

We verify this empirically: for each $n$, simulate 2,000 independent sample means and compare their variance to the theoretical $\sigma^2 / n$. Quadrupling $n$ should cut $\operatorname{Var}(\bar{x})$ by a factor of 4.

In [ ]:
sigma = 2.0   # population std, so sigma^2 = 4
for n in (4, 16, 64, 256):
    means = simulate_sample_means(rng, mu=0.0, sigma=sigma,
                                  n=n, n_trials=2000)
    print(f"n={n:4d}: empirical Var(xbar) = {means.var():.4f}   "
          f"(theory sigma^2/n = {sigma**2 / n:.4f})")

## 5. Naive Bayes spam classifier

The chapter's capstone ties everything together. For a document made of words $w_1, \dots, w_m$,

$$P(\text{class} \mid \text{words}) \;\propto\;
\underbrace{P(\text{class})}_{\text{prior (MLE count ratio)}} \;
\prod_j \underbrace{P(w_j \mid \text{class})}_{\text{likelihood (Bernoulli-style MLE)}}$$

- **Priors** $P(\text{class})$ are estimated by MLE: class counts over total documents.
- **Likelihoods** $P(\text{word} \mid \text{class})$ are MLE word frequencies with Laplace smoothing $(\text{count} + \alpha)/(\text{total} + \alpha |V|)$, so unseen words never zero out a posterior.
- The "naive" part is the **conditional independence** assumption — words are treated as independent given the class — which turns the joint likelihood into a product (a sum in log space).

We train on three spam and three ham emails, then classify a mixed message containing "free" (spammy) alongside "meeting" and "tomorrow" (hammy).

In [ ]:
spam_docs = [
    ["free", "money", "click", "now"],
    ["free", "winner", "prize", "claim"],
    ["click", "free", "offer", "limited"],
]
ham_docs = [
    ["meeting", "schedule", "tomorrow", "office"],
    ["project", "deadline", "review", "meeting"],
    ["lunch", "team", "meeting", "friday"],
]

documents = spam_docs + ham_docs
labels = ["spam"] * 3 + ["ham"] * 3

clf = NaiveBayesClassifier(alpha=1.0)   # alpha=1: Laplace smoothing
clf.fit(documents, labels)

print(f"priors: {clf.class_priors}")
print(f"P('free' | spam) = {clf.word_probs['spam']['free']:.3f}")
print(f"P('free' | ham)  = {clf.word_probs['ham']['free']:.3f}")

# Classify a new email mixing spammy and hammy words.
test_doc = ["free", "meeting", "tomorrow"]
probs = clf.predict_proba(test_doc)
print(f"\ndocument: {test_doc}")
print(f"P(spam | doc) = {probs['spam']:.3f}")
print(f"P(ham | doc)  = {probs['ham']:.3f}")
print(f"prediction: {clf.predict(test_doc)}")

## Summary

- **Bayes' theorem** combines a prior with a likelihood to get a posterior; with a 1% base rate, a positive 95%-sensitive test only yields $P(\text{disease} \mid +) \approx 0.0876$.
- **MLE for Bernoulli** is the empirical frequency $\hat{p} = k/n$ — the workhorse behind Naive Bayes parameter learning.
- The **MLE variance** (1/n) is biased low by a factor $(n-1)/n$; dividing by $n-1$ fixes it, and the difference vanishes as $n$ grows.
- **Var(x̄) = σ²/n**: averaging more samples gives proportionally more reliable estimates.
- **Naive Bayes** stitches priors, MLE likelihoods, Laplace smoothing, and conditional independence into a working spam filter.